# Step 4a: confidence head, consistency loss and stopping policy

Trains the anytime models, then scores every model, including the leak_v1 baselines, on when it guesses.
Results are on validation data; the test split is not opened. The comparisons are fixed in
`configs/anytime_v1.yaml`.

**Before running:**
1. **Add Input → Notebook output** of the leak experiment notebook (step 3). Its `runs/leak_v1` folder holds the
   baseline models; the notebook stops early if it cannot find them.
2. Attach your Kaggle Dataset and turn **Internet on**.
3. Choose **GPU T4 x2**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests/test_exp_setup.py tests/test_baseline.py tests/test_step3.py tests/test_step4.py -q",
                       shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")

# The leak_v1 baselines, from the step 3 notebook added as input.
found = [p.parent for p in Path("/kaggle/input").rglob("gru_prefix_s42/checkpoints/best.pt")]
if not found:
    raise RuntimeError("leak_v1 baselines not found: add the step 3 notebook's output as input")
BASELINES = found[0].parent.parent
print("baselines:", BASELINES, sorted(p.name for p in BASELINES.iterdir() if p.is_dir()))


In [ ]:
MAX_MINUTES = 120
!PYTHONPATH=src python -m quickdraw_exp.experiments --matrix configs/anytime_v1.yaml --max-minutes {MAX_MINUTES}
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/anytime_v1.yaml --runs /kaggle/working/runs/anytime_v1 --baselines {BASELINES} --num-workers 4


In [ ]:
import json

top = Path("/kaggle/working/runs/anytime_v1")
training = json.loads((top / "results.json").read_text())
result = json.loads((top / "anytime" / "results.json").read_text())
print("training", "COMPLETE" if training["complete"] else "INCOMPLETE, missing: " + ", ".join(training["missing"]))
print("anytime runs:", result["runs"], "missing conditions:", result["missing_conditions"])
print("\nPre-registered comparisons (area under the accuracy-earliness frontier, evaluation half):")
for row in result["comparisons"]:
    if row["difference"] is None:
        print(f"  {row['name']}: not available")
    else:
        print(f"  {row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
              f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")
print("\nPer condition and policy:")
for condition, policies in result["table"].items():
    for policy, e in policies.items():
        extra = ""
        if "pictionary" in e:
            extra = (f"  pictionary {e['pictionary']['mean']:.4f}  acc@0.3 {e['accuracy_at_0.3']['mean']:.4f}"
                     f"  acc@0.5 {e['accuracy_at_0.5']['mean']:.4f}  earliness@gap2% {e['earliness_at_gap_0.02']['mean']:.3f}"
                     f"  gap2% met {e['met_gap_0.02']}  earliness@gap5% {e['earliness_at_gap_0.05']['mean']:.3f}  gap5% met {e['met_gap_0.05']}")
        print(f"  {condition:20s} {policy:13s} auc {e['auc']['mean']:.4f} +- {e['auc']['std']:.4f}{extra}")
print("\nReference points:")
for run, ref in result["reference"].items():
    print(f"  {run}: end {ref['end_of_drawing']['accuracy']:.4f}, first check {ref['first_checkpoint']['accuracy']:.4f} at "
          f"{ref['first_checkpoint']['earliness']:.3f}, oracle {ref['oracle_first_correct']['accuracy']:.4f} at "
          f"{ref['oracle_first_correct']['earliness']:.3f}")
for name in training["missing"]:
    log = top / f"{name}.log"
    if log.exists():
        print(f"--- {name} ---")
        print("\n".join(log.read_text().splitlines()[-15:]))
